In [120]:
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
from tqdm.auto import tqdm

In [121]:
DATA_ROOT = Path("D:/Almaz/ML_Ваттметрирование/Datasets")
OUTPUT_ROOT = DATA_ROOT / "parquet"
OUTPUT_ROOT.mkdir(exist_ok=True)
TEST_DIR = DATA_ROOT / "test"

In [122]:
OPTUNA_N_TRIALS = 15

In [123]:
FILENAME_RE = re.compile(
    r"^(?P<speed_code>N\d{2})_(?P<load_code>M\d{2})_(?P<force_code>F\d{2})_"
    r"(?P<bearing_code>[A-Z]{1,2}\d{2,3})_(?P<run>\d+)$"
)

SPEED_RPM = {"N09": 900, "N15": 1500}
LOAD_TORQUE_NM = {"M07": 0.7, "M01": 0.1}
RADIAL_FORCE_N = {"F10": 1000, "F04": 400}

Исходные 6 классов

In [124]:
# TARGET_MAPPING = {
#     **{code: "healthy" for code in ["K001", "K002", "K003", "K004", "K005", "K006"]},
#     **{
#         code: "artificial_damage_outer_ring"
#         for code in ["KA01", "KA03", "KA05", "KA06", "KA07", "KA08", "KA09"]
#     },
#     **{
#         code: "artificial_damage_inner_ring"
#         for code in ["KI01", "KI03", "KI05", "KI07", "KI08"]
#     },
#     **{
#         code: "real_damage_outer_ring"
#         for code in ["KA04", "KA15", "KA16", "KA22", "KA30"]
#     },
#     **{
#         code: "real_damage_inner_ring"
#         for code in ["KI04", "KI14", "KI16", "KI17", "KI18", "KI21"]
#     },
#     **{code: "real_damage_both_rings" for code in ["KB23", "KB24", "KB27"]},
# }

3 класса - здоровые, повреждения внешнего и внутреннего кольца

In [125]:
TARGET_MAPPING = {
    **{code: "healthy" for code in ["K001", "K002", "K003", "K004", "K005", "K006"]},
    **{
        code: "damage_outer_ring"
        for code in [
            "KA01",
            "KA03",
            "KA05",
            "KA06",
            "KA07",
            "KA08",
            "KA09",
            "KA04",
            "KA15",
            "KA16",
            "KA22",
            "KA30",
        ]
    },
    **{
        code: "damage_inner_ring"
        for code in [
            "KI01",
            "KI03",
            "KI05",
            "KI07",
            "KI08",
            "KI04",
            "KI14",
            "KI16",
            "KI17",
            "KI18",
            "KI21",
        ]
    },
    **{code: "damage_both_rings" for code in ["KB23", "KB24", "KB27"]},
}

3 класса - здоровые, повреждения искусственные и реальные

In [126]:
# TARGET_MAPPING = {
#     **{code: "healthy" for code in ["K001", "K002", "K003", "K004", "K005", "K006"]},
#     **{
#         code: "artificial_damage"
#         for code in ["KA01", "KA03", "KA05", "KA06", "KA07", "KA08", "KA09", "KI01", "KI03", "KI05", "KI07", "KI08"]
#     },
#     **{
#         code: "real_damage"
#         for code in ["KA04", "KA15", "KA16", "KA22", "KA30", "KI04", "KI14", "KI16", "KI17", "KI18", "KI21", "KB23", "KB24", "KB27"]
#     }
# }

In [127]:
def bearing_condition(bearing_code: str) -> str:
    return TARGET_MAPPING[bearing_code]


def parse_filename(stem: str) -> dict:
    m = FILENAME_RE.match(stem)
    if not m:
        raise ValueError(f"Filename does not match the expected pattern: {stem!r}")
    g = m.groupdict()
    for code, table, label in (
        (g["speed_code"], SPEED_RPM, "speed_code"),
        (g["load_code"], LOAD_TORQUE_NM, "load_code"),
        (g["force_code"], RADIAL_FORCE_N, "force_code"),
    ):
        if code not in table:
            raise ValueError(f"Unknown {label} {code!r} in {stem!r}")
    return {
        "file_stem": stem,
        "speed_rpm": SPEED_RPM[g["speed_code"]],
        "load_torque_nm": LOAD_TORQUE_NM[g["load_code"]],
        "radial_force_n": RADIAL_FORCE_N[g["force_code"]],
        "bearing_code": g["bearing_code"],
        "bearing_condition": bearing_condition(g["bearing_code"]),
        "run": int(g["run"]),
    }

In [128]:
def _slug(raster: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", raster.lower()).strip("_")


def load_mat_file(path: Path) -> tuple[dict[str, pd.DataFrame], dict]:
    """Load one Paderborn .mat measurement into {raster_name: DataFrame} + metadata."""
    raw = sio.loadmat(path, struct_as_record=False, squeeze_me=True)
    key = next(k for k in raw if not k.startswith("__"))
    top = raw[key]

    time_by_raster = {
        x.Raster: np.asarray(x.Data, dtype=float) for x in np.atleast_1d(top.X)
    }

    channels_by_raster: dict[str, dict[str, np.ndarray]] = {}
    for ch in np.atleast_1d(top.Y):
        channels_by_raster.setdefault(ch.Raster, {})[ch.Name] = np.asarray(
            ch.Data, dtype=float
        )

    frames = {}
    for raster, channels in channels_by_raster.items():
        df = pd.DataFrame(channels)
        time = time_by_raster.get(raster)
        if time is not None and len(time) == len(df):
            df.insert(0, "time_s", time)
        frames[raster] = df

    metadata = {
        "measurement_id": int(top.Info.MeasurementID),
        "recording_datetime": datetime.strptime(
            str(top.Description.General.DateTime), "%d.%m.%Y %H:%M:%S"
        ),
        "measurement_length_s": float(top.Description.Measurement.Length),
    }
    return frames, metadata

In [129]:
# _sample_path = DATA_ROOT / "K001" / "N09_M07_F10_K001_1.mat"
# _frames, _meta = load_mat_file(_sample_path)

# print(_meta)
# for raster, df in _frames.items():
#     print(raster, df.shape, list(df.columns))
# _frames["HostService"].head()

In [130]:
def convert_and_save(mat_path: Path, output_root: Path) -> dict:
    """Convert one .mat file to per-raster Parquet files and return a catalog row."""
    mat_path = Path(mat_path)
    meta = parse_filename(mat_path.stem)
    frames, mat_meta = load_mat_file(mat_path)

    bearing_dir = output_root / meta["bearing_code"]
    bearing_dir.mkdir(parents=True, exist_ok=True)

    row = {**meta, **mat_meta, "source_file": str(mat_path)}
    for raster, df in frames.items():
        slug = _slug(raster)
        out_path = bearing_dir / f"{mat_path.stem}__{slug}.parquet"
        df.to_parquet(out_path, index=False)
        row[f"n_samples_{slug}"] = len(df)
        row[f"path_{slug}"] = str(out_path.relative_to(output_root))
    return row


def discover_mat_files(data_root: Path) -> list[Path]:
    return sorted(data_root.glob("K*/*.mat"))


def build_catalog(mat_files: list[Path], output_root: Path) -> pd.DataFrame:
    rows, errors = [], []
    for f in tqdm(mat_files):
        try:
            rows.append(convert_and_save(f, output_root))
        except Exception as exc:
            errors.append((str(f), repr(exc)))
    if errors:
        print(f"{len(errors)} file(s) failed:")
        for f, e in errors:
            print(" ", f, "->", e)
    return pd.DataFrame(rows)

Читаем директории и конвертируем .mat в .parquet


In [131]:
def prepare_metadata(data_dir: Path, out_dir: Path, filename: str):
    path = out_dir / filename
    if path.is_file():
        return pd.read_parquet(path)

    all_mat_files = discover_mat_files(data_dir)
    catalog = build_catalog(all_mat_files, out_dir)
    catalog.to_parquet(path, index=False)
    return catalog

In [132]:
catalog = prepare_metadata(DATA_ROOT, OUTPUT_ROOT, "catalog.parquet")
catalog.head()

  0%|          | 0/1920 [00:00<?, ?it/s]

,file_stem,speed_rpm,load_torque_nm,radial_force_n,bearing_code,bearing_condition,run,measurement_id,recording_datetime,measurement_length_s,source_file,n_samples_mech_4khz,path_mech_4khz,n_samples_hostservice,path_hostservice,n_samples_temp_1hz,path_temp_1hz
0,N09_M07_F10_K001_1,900,0.7,1000,K001,healthy,1,22,2015-09-17 19:51:22,4.0,D:\Almaz\ML_Ваттметрирование\Datasets\K001\N09...,16008,K001\N09_M07_F10_K001_1__mech_4khz.parquet,256823,K001\N09_M07_F10_K001_1__hostservice.parquet,5,K001\N09_M07_F10_K001_1__temp_1hz.parquet
1,N09_M07_F10_K001_10,900,0.7,1000,K001,healthy,10,31,2015-09-17 20:00:03,4.0,D:\Almaz\ML_Ваттметрирование\Datasets\K001\N09...,16007,K001\N09_M07_F10_K001_10__mech_4khz.parquet,256608,K001\N09_M07_F10_K001_10__hostservice.parquet,5,K001\N09_M07_F10_K001_10__temp_1hz.parquet
2,N09_M07_F10_K001_11,900,0.7,1000,K001,healthy,11,32,2015-09-17 20:00:37,4.0,D:\Almaz\ML_Ваттметрирование\Datasets\K001\N09...,16007,K001\N09_M07_F10_K001_11__mech_4khz.parquet,256096,K001\N09_M07_F10_K001_11__hostservice.parquet,5,K001\N09_M07_F10_K001_11__temp_1hz.parquet
3,N09_M07_F10_K001_12,900,0.7,1000,K001,healthy,12,33,2015-09-17 20:01:10,4.0,D:\Almaz\ML_Ваттметрирование\Datasets\K001\N09...,16001,K001\N09_M07_F10_K001_12__mech_4khz.parquet,256000,K001\N09_M07_F10_K001_12__hostservice.parquet,5,K001\N09_M07_F10_K001_12__temp_1hz.parquet
4,N09_M07_F10_K001_13,900,0.7,1000,K001,healthy,13,34,2015-09-17 20:01:41,4.0,D:\Almaz\ML_Ваттметрирование\Datasets\K001\N09...,16001,K001\N09_M07_F10_K001_13__mech_4khz.parquet,256000,K001\N09_M07_F10_K001_13__hostservice.parquet,5,K001\N09_M07_F10_K001_13__temp_1hz.parquet


In [133]:
BALL_PARAMS = dict(d_ball=6.75, d_pitch=28.55, n_balls=8, phi=0, pole_pairs=4)
FS = 64000.0
HARMONICS = [1, 2, 3]
# TARGET_NAME_TO_CODE = {
#     "healthy": 0,
#     "artificial_damage_outer_ring": 1,
#     "real_damage_outer_ring": 2,
#     "real_damage_both_rings": 3,
#     "artificial_damage_inner_ring": 4,
#     "real_damage_inner_ring": 5,
# }
TARGET_NAME_TO_CODE = {
    "healthy": 0,
    "damage_inner_ring": 1,
    "damage_outer_ring": 2,
}
# N_CLASSES = 6
N_CLASSES = 3

In [134]:
def bearing_freqs():
    r = BALL_PARAMS["d_ball"] / BALL_PARAMS["d_pitch"] * np.cos(BALL_PARAMS["phi"])
    n = BALL_PARAMS["n_balls"]
    return {
        "BPFO": n / 2 * (1 - r),
        "BPFI": n / 2 * (1 + r),
        "BSF": BALL_PARAMS["d_pitch"] / (2 * BALL_PARAMS["d_ball"]) * (1 - r**2),
        "FTF": 0.5 * (1 - r),
    }

In [135]:
import scipy
import polars as pl

In [136]:
def fourier_transform_vibration(frame: pd.Series, fs: float):
    """Преобразование Фурье для сигналов вибрации"""
    x = frame.to_numpy()
    ny = fs / 2
    band = [2000, ny * 0.95]

    sos = scipy.signal.butter(4, band, btype="bandpass", fs=fs, output="sos")
    x_filt = scipy.signal.sosfiltfilt(sos, (x - x.mean()))
    env = np.abs(scipy.signal.hilbert(x_filt))
    env -= env.mean()

    window = scipy.signal.windows.hann(len(env))
    fft_values = scipy.fft.rfft(env * window)
    freqs = scipy.fft.rfftfreq(len(env), 1 / fs)
    amps = np.abs(fft_values) * 2 / np.sum(window)
    return dict(frequences=freqs, amplitudes=amps)

In [137]:
def fourier_transform_current(frame: pd.Series, fs: float):
    """Преобразование Фурье для сигналов тока"""
    x = frame.to_numpy()

    window = scipy.signal.windows.hann(len(x))
    fft_values = scipy.fft.rfft((x - x.mean()) * window)
    freqs = scipy.fft.rfftfreq(len(x), 1 / fs)
    amps = np.abs(fft_values) * 2 / np.sum(window)
    return dict(frequences=freqs, amplitudes=amps)

In [138]:
def peak_amplitude(freqs, amps, f0, tol):
    if f0 <= 0 or f0 >= freqs[-1]:
        return 0.0
    mask = (freqs >= f0 - tol) & (freqs <= f0 + tol)
    return amps[mask].max()

In [139]:
def prepare_time_features(frame: pd.DataFrame, agg_cols: list[str]):
    N = frame.shape[0] // 4
    features = {}
    time_df = pd.DataFrame(index=frame.index)
    for col in agg_cols:
        time_df[f"{col}_kurtosis"] = frame[f"{col}"].rolling(N).kurt()
        time_df[f"{col}_mean"] = frame[f"{col}"].rolling(N).mean()
        time_df[f"{col}_min"] = frame[f"{col}"].rolling(N).min()
        time_df[f"{col}_max"] = frame[f"{col}"].rolling(N).max()
        time_df[f"{col}_std"] = frame[f"{col}"].rolling(N).std()
        time_df[f"{col}_skew"] = frame[f"{col}"].rolling(N).skew()

    time_df = time_df.dropna()
    values_per_second = time_df.iloc[::N].iloc[1:]
    for col in time_df.columns:
        for t, val in enumerate(values_per_second[col]):
            features[f"{col}__t{(t+1)}"] = val

    return features

Итеративно обрабатываем метадатасет и применяем преобразования...


In [140]:
import pywt


def prepare_wpd_features(
    df: pd.DataFrame, prefix: str, level: int = 3, wavelet: str = "db4"
) -> dict:
    x = df[prefix].to_numpy()
    """Нормированные энергии вейвлет-пакетного разложения"""

    wp = pywt.WaveletPacket(data=x, wavelet=wavelet, mode="symmetric", maxlevel=level)
    nodes = [n.path for n in wp.get_level(level, "natural")]
    e = np.array([np.sum(wp[p].data ** 2) for p in nodes], dtype=np.float64)
    e = e / (e.sum() + 1e-20)
    return {f"{prefix}_wpd{i}": float(v) for i, v in enumerate(e)}

In [141]:
def welch_psd(x: np.ndarray, fs: float, nperseg: int = 4096):
    f, p = scipy.signal.welch(x, fs=fs, nperseg=min(nperseg, len(x)), scaling="density")
    return f, p + 1e-20


def prepare_spectral_features(df: pd.DataFrame, fs: float, prefix: str, n_bands: int = 10) -> dict:
    """Логарифмические энергии в полосах"""
    x = df[prefix].to_numpy()
    f, p = welch_psd(x, fs)
    pn = p / p.sum()
    out = {
        f"{prefix}_sp_centroid": float((f * pn).sum()),
        f"{prefix}_sp_spread": float(np.sqrt((((f - (f * pn).sum()) ** 2) * pn).sum())),
        f"{prefix}_sp_entropy": float(-(pn * np.log(pn)).sum() / np.log(len(pn))),
    }

    # лог. сетка от 10 Гц до частоты Найквиста
    edges = np.logspace(np.log10(10.0), np.log10(fs / 2), n_bands + 1)
    for i in range(n_bands):
        m = (f >= edges[i]) & (f < edges[i + 1])
        out[f"{prefix}_band{i}"] = (
            float(np.log10(p[m].sum() + 1e-20)) if m.any() else -20.0
        )
    return out

In [142]:
def prepare_feature_spectrum(df: pd.DataFrame, prefix: str, rpm: int):
    """
    Готовим спектральные признаки по каждой заданной в словаре частоте.
    - prefix - имя колонки, ток или вибрация
    - bearing_features - словарь с нужными частотами, на которых ищем аномалии, причем базовая частота идет как скорость вращения вала / 60
    - harmonics - нужная нам гармоника
    - rpm - частота вращения
    - f_e - частота питания
    - fr - частота вращения / 60
    - tol - допущение, в пределах которого искомая частота
    - f0 - коэффициент нужной частоты (bpfi, bpfo, ftf, bsf)
    """
    fr = rpm / 60
    features = {}
    bearing_features = bearing_freqs().items()
    if prefix.startswith("vibration"):
        # вибрация
        fourier_signal = fourier_transform_vibration(df[prefix], FS)
        freqs = fourier_signal["frequences"]
        amps = fourier_signal["amplitudes"]

        floor = np.median(amps[(freqs > 5) & (freqs < 1000)]) + 1e-12
        tol = max(2.0, 3 * (freqs[1] - freqs[0]))

        # гармоники
        for name, f0 in bearing_features:
            for h in HARMONICS:
                features[f"{prefix}_{name}_h{h}"] = (
                    peak_amplitude(freqs, amps, h * f0 * fr, tol) / floor
                )

        # боковые полосы
        for s in (-1, 1):
            tag = "p" if s > 0 else "m"
            features[f"{prefix}_BPFI_{tag}"] = (
                peak_amplitude(freqs, amps, bearing_freqs()["BPFI"] * fr + s * fr, tol)
                / floor
            )

        # частота вращения
        features[f"{prefix}_env_fr"] = peak_amplitude(freqs, amps, fr, tol)
        # features[f"{prefix}_env_rms"] = float(np.sqrt(np.mean(env**2)))
    else:
        # токи
        fourier_signal = fourier_transform_current(df[prefix], FS)
        freqs = fourier_signal["frequences"]
        amps = fourier_signal["amplitudes"]
        f_e = fr * BALL_PARAMS["pole_pairs"]
        tol = max(2.0, 3 * (freqs[1] - freqs[0]))

        # частота несущая
        f_e_amp = peak_amplitude(freqs, amps, f_e, tol) + 1e-12
        features[f"{prefix}_fe_amp"] = f_e_amp
        for name, f0 in bearing_features:
            for s in (-1, 1):
                tag = "p" if s == 1 else "m"
                features[f"{prefix}_{name}_{tag}"] = (
                    peak_amplitude(freqs, amps, f_e + s * f0 * fr, tol) / f_e_amp
                )

        # гармоники питания
        for k in (2, 3, 5):
            features[f"{prefix}_fe_h{k}"] = peak_amplitude(freqs, amps, k * f_e, tol)

    return features

In [143]:
def prepare_hostservice(path: str, rpm: int, agg_cols: list[str]):
    features = {}
    frame = pd.read_parquet(path)
    for i in [1, 2]:
        spectrum = prepare_feature_spectrum(frame, f"phase_current_{i}", rpm)
        features.update(spectrum)
        features.update(prepare_wpd_features(frame, f"phase_current_{i}"))
        features.update(prepare_spectral_features(frame, FS, f"phase_current_{i}"))

    features.update(prepare_feature_spectrum(frame, "vibration_1", rpm))
    features.update(prepare_spectral_features(frame, FS, "vibration_1"))
    features.update(prepare_wpd_features(frame, "vibration_1"))
    features.update(prepare_time_features(frame, agg_cols))
    return features

In [144]:
def preprocessing(df: pd.DataFrame, save_path: Path, generate_new: bool = False):
    if generate_new and save_path.is_file():
        return pd.read_csv(save_path)

    cols = [
        "speed_rpm",
        "load_torque_nm",
        "radial_force_n",
        "bearing_condition",
        "measurement_id",
        "path_hostservice",
        "bearing_code",
    ]
    result = df.copy().loc[:, cols]

    aggregate_cols = ["phase_current_1", "phase_current_2", "vibration_1"]
    feature_rows = []
    for _, row in tqdm(df.iterrows()):
        features = {}

        current_data_path = OUTPUT_ROOT / row["path_hostservice"]
        features.update(
            prepare_hostservice(current_data_path, row["speed_rpm"], aggregate_cols)
        )

        feature_rows.append(features)

    features_df = pd.DataFrame(feature_rows, index=result.index)
    result = pd.concat([result, features_df], axis=1)
    result.to_csv(save_path)
    # current_columns = [col for col in result.columns if "current" in col]
    # result[current_columns] = result[current_columns].ffill()
    # result = result.sort_values(by="measurement_id")
    return result

In [145]:
preprocessed_df = preprocessing(
    catalog, DATA_ROOT / "full_prepared.csv", generate_new=False
)

0it [00:00, ?it/s]

In [146]:
preprocessed_df

,speed_rpm,load_torque_nm,radial_force_n,bearing_condition,measurement_id,path_hostservice,bearing_code,phase_current_1_fe_amp,phase_current_1_BPFO_m,phase_current_1_BPFO_p,...,vibration_1_min__t3,vibration_1_max__t1,vibration_1_max__t2,vibration_1_max__t3,vibration_1_std__t1,vibration_1_std__t2,vibration_1_std__t3,vibration_1_skew__t1,vibration_1_skew__t2,vibration_1_skew__t3
0,900,0.7,1000,healthy,22,K001\N09_M07_F10_K001_1__hostservice.parquet,K001,2.193268,0.000262,0.000277,...,-4.843140,3.771973,4.690552,4.141235,0.364229,0.365423,0.367673,-0.024172,0.087590,-0.048972
1,900,0.7,1000,healthy,31,K001\N09_M07_F10_K001_10__hostservice.parquet,K001,2.052334,0.000468,0.000306,...,-3.833008,4.037476,4.000854,4.049683,0.363079,0.361361,0.363440,-0.026363,0.053745,0.006471
2,900,0.7,1000,healthy,32,K001\N09_M07_F10_K001_11__hostservice.parquet,K001,2.410373,0.000264,0.000238,...,-3.833008,4.455566,3.866577,4.104614,0.360117,0.363880,0.362216,0.071844,-0.021445,0.067747
3,900,0.7,1000,healthy,33,K001\N09_M07_F10_K001_12__hostservice.parquet,K001,2.404953,0.000346,0.000238,...,-5.166626,3.747559,4.055786,4.180908,0.360075,0.360837,0.360692,0.065863,0.043941,0.040132
4,900,0.7,1000,healthy,34,K001\N09_M07_F10_K001_13__hostservice.parquet,K001,2.407388,0.000287,0.000307,...,-3.909302,4.183960,3.311157,4.431152,0.360323,0.365994,0.359603,0.212147,0.093068,0.266342
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1915,1500,0.7,1000,damage_inner_ring,13,KI21\N15_M07_F10_KI21_5__hostservice.parquet,KI21,2.334538,0.000193,0.000189,...,-1.629639,1.477051,1.440430,1.538086,0.285370,0.286223,0.286506,-0.009478,-0.011467,-0.007427
1916,1500,0.7,1000,damage_inner_ring,14,KI21\N15_M07_F10_KI21_6__hostservice.parquet,KI21,2.388890,0.000141,0.000229,...,-1.373291,1.730347,1.544189,1.580811,0.286606,0.281261,0.281030,-0.016216,0.018106,-0.046623
1917,1500,0.7,1000,damage_inner_ring,15,KI21\N15_M07_F10_KI21_7__hostservice.parquet,KI21,2.550899,0.000233,0.000217,...,-1.544189,1.452637,1.406860,1.562500,0.281159,0.288552,0.287499,-0.007959,-0.013151,-0.004016
1918,1500,0.7,1000,damage_inner_ring,17,KI21\N15_M07_F10_KI21_8__hostservice.parquet,KI21,2.513107,0.000204,0.000165,...,-1.370239,1.516724,1.583862,1.608276,0.287711,0.285426,0.285444,0.002513,-0.043005,-0.010792


In [147]:
def prepare_target(df: pd.DataFrame):
    X = df.drop(
        [
            "measurement_id",
            "bearing_condition",
            "bearing_code",
            "path_hostservice",
            # "speed_rpm",
            # "radial_force_n",
            # "load_torque_nm",
        ],
        axis=1,
    )
    y = df["bearing_condition"].map(TARGET_NAME_TO_CODE)

    groups = df["bearing_code"]
    return X, y, groups

In [148]:
from sklearn.preprocessing import StandardScaler


def scale_features(X, scaler=None):
    X_arr = (
        X.to_numpy(dtype=np.float32)
        if hasattr(X, "to_numpy")
        else np.asarray(X, dtype=np.float32)
    )

    if scaler is None:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_arr)
    else:
        X_scaled = scaler.transform(X_arr)

    return X_scaled, scaler

In [149]:
from sklearn.model_selection import StratifiedGroupKFold


def train_test_split(
    X: pd.DataFrame,
    y: np.ndarray,
    groups: pd.Series,
    n_splits=2,
    test_size=0.3,
    random_state=42,
):
    splitter = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )
    return splitter.split(X, y, groups=groups)

In [150]:
X, y, groups = prepare_target(preprocessed_df)
train_test_idx = train_test_split(X, y, groups, n_splits=OPTUNA_N_TRIALS)

In [151]:
import json
import warnings

import xgboost as xgb


def select_xgboost_device() -> str:
    """Select CUDA only after XGBoost successfully trains on the GPU."""
    if not xgb.build_info().get("USE_CUDA", False):
        print("CUDA unavailable: XGBoost was built without CUDA support.")
        return "cpu"

    probe_data = xgb.DMatrix(
        np.array([[0.0], [1.0], [2.0], [3.0]], dtype=np.float32),
        label=np.array([0, 0, 1, 1], dtype=np.float32),
    )
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            probe = xgb.train(
                {"device": "cuda", "tree_method": "hist", "objective": "binary:logistic"},
                probe_data,
                num_boost_round=1,
            )
        actual_device = json.loads(probe.save_config())["learner"]["generic_param"]["device"]
        if actual_device.startswith("cuda"):
            return "cuda"
        reason = "; ".join(str(item.message) for item in caught)
        print(f"CUDA unavailable: {reason or 'XGBoost selected CPU.'}")
    except xgb.core.XGBoostError as exc:
        print(f"CUDA probe failed; using CPU: {exc}")
    return "cpu"


DEVICE = select_xgboost_device()
print(f"XGBoost DEVICE = {DEVICE}")

# Shared by Optuna trials and the final model. Feature extraction runs on CPU.
XGB_BASE_PARAMS = {
    "device": DEVICE,
    "tree_method": "hist",
    "objective": "multi:softprob",
    "n_jobs": 1,
    "random_state": 42,
}


CUDA unavailable: [11:33:46] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:55: No visible GPU is found, setting device to CPU.; [11:33:46] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
XGBoost DEVICE = cpu


In [152]:
from sklearn.metrics import f1_score
from xgboost import XGBClassifier


def make_objective(train_test_idx):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 500, step=25),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", 2, 6),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 8),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_categorical("reg_alpha", [0.0, 0.01, 0.1, 1.0]),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 20.0, log=True),
        }
        train_idx, test_idx = next(train_test_idx)
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[test_idx], y.iloc[test_idx]

        model = XGBClassifier(**XGB_BASE_PARAMS, **params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val).ravel()
        return f1_score(y_val, y_pred, average="macro")

    return objective

In [153]:
import optuna

objective = make_objective(train_test_idx)
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=OPTUNA_N_TRIALS)

[I 2026-11-19 11:33:46,375] A new study created in memory with name: no-name-db96ea5b-1187-4d62-9574-3df4d8313c42
D:\Almaz\conda\Lib\site-packages\sklearn\externals\array_api_compat\numpy\_aliases.py:106: RuntimeWarning: invalid value encountered in cast
  return x.astype(dtype=dtype, copy=copy)
[W 2026-11-19 11:33:46,379] Trial 0 failed with parameters: {'n_estimators': 250, 'learning_rate': 0.19143207824094388, 'max_depth': 6, 'min_child_weight': 8, 'subsample': 0.6175788968268401, 'colsample_bytree': 0.6922483618930106, 'reg_alpha': 0.01, 'reg_lambda': 0.012579105440180174} because of the following error: ValueError('Input contains NaN.').
Traceback (most recent call last):
  File "D:\Almaz\conda\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\User\AppData\Local\Temp\ipykernel_29632\1000842060.py", line 17, in objective
    train_idx, test_idx = next(train_test_idx)
                          ~~~~^^^^^^^^^^^^^^^

ValueError: Input contains NaN.

In [ ]:
best_model = XGBClassifier(**XGB_BASE_PARAMS, **study.best_params)
best_model.fit(X, y)

,"objective objective: typing.Union[str, xgboost.objective.Objective, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.9601419036686308
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cpu'
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: pyt

### Инференс


In [ ]:
best_model.save_model("best_model.json")

In [156]:
test_catalog = prepare_metadata(TEST_DIR, OUTPUT_ROOT, "test_catalog.parquet")

  0%|          | 0/560 [00:00<?, ?it/s]

In [157]:
test_df = preprocessing(test_catalog, DATA_ROOT / "test" / "test.csv")
X_true, y_true, _ = prepare_target(test_df)

0it [00:00, ?it/s]

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)
import matplotlib.pyplot as plt

y_pred = best_model.predict(X_true).ravel()

print(f"Accuracy = {accuracy_score(y_true, y_pred):.3f}")
print(f"Macro F1 : {f1_score(y_true, y_pred, average='macro'):.3f}\n")
print(classification_report(y_true, y_pred, zero_division=0))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    normalize="true",
    values_format=".2f",
    xticks_rotation=45,
    cmap="Blues",
    colorbar=False,
    ax=ax,
)
ax.set_title("Матрица ошибок (доли от истинного класса)")
plt.tight_layout()
plt.show()